# Week 12 — Business-Process Optimization: Support-Ticket Lifecycle

Diagnoses where InGen's fleet-support cycle time goes and quantifies what operational changes would buy.

**Data:** the Week 7 synthetic warehouse. Week 7's `fact_support_tickets` has no stages, teams or workload — all three are required here — so the lifecycle layer is generated by a documented discrete-event model (`src/process/process_model.py`) that **replays Week 7's real ticket stream**. Every generated ticket joins 1:1 back to a real Week 7 ticket_key, and stage durations reconcile exactly to each ticket's cycle time.

**Honesty:** every number here is a property of that model, not a measurement of InGen's support org. What transfers is the validated method + reusable pipeline.

## 1. Generate the stage-level events (and reconcile to the Week 7 schema)

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable,'-m','src.process.stage_generator'],capture_output=True,text=True,cwd='../..').stdout)

## 2. Cycle-time decomposition, Pareto and control chart

In [ ]:
print(subprocess.run([sys.executable,'-m','src.process.cycle_time'],capture_output=True,text=True,cwd='../..').stdout)

### Pareto — top 3 stages are 90% of all process time

In [ ]:
from IPython.display import Image
Image('../../reports/week12/pareto_stages.png')

### The chart that drives the recommendation: waiting vs staffed work

Headcount can only shorten the blue (staffed) portion. Parts & dispatch is 100% orange — an unstaffed wait.

In [ ]:
Image('../../reports/week12/wait_vs_service.png')

## 3. Driver analysis — regression with 95% confidence intervals

Plus a **method validation**: because the data came from a model with known parameters, the regression should recover their direction and ordering.

In [ ]:
print(subprocess.run([sys.executable,'-m','src.process.drivers'],capture_output=True,text=True,cwd='../..').stdout)

## 4. Capacity simulation — 5 scenarios x 8 replications, paired deltas

In [ ]:
print(subprocess.run([sys.executable,'-m','src.process.capacity_sim'],capture_output=True,text=True,cwd='../..').stdout)

In [ ]:
Image('../../reports/week12/scenario_comparison.png')

## 5. Build the 3-page memo

In [ ]:
print(subprocess.run([sys.executable,'-m','src.process.build_report'],capture_output=True,text=True,cwd='../..').stdout)

## 6. Takeaways
- **Parts & dispatch is 53% of all process time and 100% waiting** — no amount of headcount touches it.
- The brief's drivers (product, severity, geography, weekday, workload) explain only **4%** of cycle-time variance. Add one structural fact — did the ticket need a part? — and R² jumps to **72%** (+578% on cycle time).
- **Regional parts buffer: −23%** mean cycle time (P90 64h→47h). **+1 Field FTE: −8%**. **Both: −29%**.
- **Rerouting Tier-1 triage — the obvious lever — does nothing** (−1.9%, CI spans zero). Triage is 4% of process time.
- Amdahl's law in practice: +1 Field FTE nearly zeroes field queue wait (4.9h→0.4h) but buys only 8%, because that stage is ~15% of the process.
- The process is broadly **in statistical control** — long cycle times are by design, not anomalies to firefight.